# Thư viện và dữ liệu

In [1]:
#thư viện
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from sklearn.model_selection import ParameterGrid

In [1]:
#tải dữ liệu
df = pd.read_csv('../tnbike_data.csv', parse_dates=['ds'])
df = df.rename(columns={'revenue': 'y'})
df.head()

In [1]:
#biến ngày
df['ds'] = pd.to_datetime(df['ds'])
df.ds

# Ngày lễ

In [1]:
#Tết Nguyên Đán
dates_tet = pd.to_datetime(df[df.is_tet == 1].ds)
tet = pd.DataFrame({'holiday': 'tet_nguyen_dan', 'ds': dates_tet,
                    'lower_window': -7, 'upper_window': 7})

In [1]:
#Ngày Thiếu nhi 1/6
dates_thieu_nhi = pd.to_datetime(df[df.is_thieu_nhi == 1].ds)
thieu_nhi = pd.DataFrame({'holiday': 'ngay_thieu_nhi', 'ds': dates_thieu_nhi,
                          'lower_window': -14, 'upper_window': 3})

In [1]:
#gộp ngày lễ
holidays = pd.concat([tet, thieu_nhi])
holidays

In [1]:
#bỏ cột ngày lễ
df = df.drop(['is_tet', 'is_thieu_nhi'], axis=1)

In [1]:
#tách train
training = df.iloc[:-3, :]
future_df = df.iloc[-3:, :]

# Lưới tham số

In [1]:
#định nghĩa lưới
param_grid = {'changepoint_prior_scale': [0.01, 0.05, 0.1, 0.5],
              'holidays_prior_scale':    [0.01, 1.0, 10.0],
              'seasonality_prior_scale': [0.01, 1.0, 10.0],
              'seasonality_mode':        ['additive', 'multiplicative']}
grid = list(ParameterGrid(param_grid))
print(f'Tổng tổ hợp: {len(grid)}')

In [1]:
#chạy cross-validation
rmse = []
for params in grid:
    m = Prophet(holidays=holidays, **params)
    m.fit(training)
    df_cv = cross_validation(m, horizon='90 days', period='30 days', initial='90 days')
    df_p  = performance_metrics(df_cv, rolling_window=1)
    rmse.append(df_p['rmse'].values[0])

# Xuất kết quả

In [1]:
#kết quả
tuning_results = pd.DataFrame(grid)
tuning_results['rmse'] = rmse
tuning_results

In [1]:
#lưu tham số tốt nhất
best_params = tuning_results[tuning_results.rmse == tuning_results.rmse.min()].transpose()
best_params.to_csv('../Forecasting Product/best_params_prophet.csv')
best_params